<a href="https://colab.research.google.com/github/ngonhatanhly-NNA/AI-Training/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.utils.data import Dataset

class BiosDataset(Dataset):
    def __init__(self, texts, professions, genders, vectorizer=None):
        self.features = vectorizer.transform(texts).toarray()
        self.labels = professions
        self.genders = genders

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        g = torch.tensor(self.genders[idx], dtype=torch.long)
        return x, y, g

In [ ]:
import torch.nn as nn

class ProfessionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(ProfessionClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # Kĩ thuật dropout để giảm overfitting, ngẫu nhiên tắ 30% nơ ron
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

print ("Loading data....")
train_df = pd.read_csv("biosbias_train_cleaned.csv")
test_df = pd.read_csv("biosbias_test_cleaned.csv")

# Handle null in processing data
train_df = train_df.dropna(subset=['hard_text', 'profession'])
test_df = test_df.dropna(subset=['hard_text', 'profession', 'gender'])
# Vecotorize the text data using TF-IDF
print( "Vectorizing text data....")
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_df['hard_text'])
X_test_tfidf = vectorizer.transform(test_df['hard_text'])

# Label encode the target variable
print( "Encoding labels....")
label_encoder_prof = LabelEncoder()
y_train_encoded = label_encoder_prof.fit_transform(train_df['profession'])
y_test_encoded = label_encoder_prof.transform(test_df['profession'])

label_encoder_gender = LabelEncoder()
gender_train_encoded = label_encoder_gender.fit_transform(train_df['gender'])
gender_test_encoded = label_encoder_gender.transform(test_df['gender'])

print( "Creating datasets....")

In [ ]:
from torch.utils.data import DataLoader

# Khởi tạo Dataset từ class BiosDataset mà bạn đã định nghĩa ở Cell 1
train_dataset = BiosDataset(train_df["hard_text"], y_train_encoded, gender_train_encoded, vectorizer=vectorizer)
test_dataset = BiosDataset(test_df["hard_text"], y_test_encoded, gender_test_encoded, vectorizer=vectorizer)

# Bọc bằng DataLoader để tự động chia nhỏ dữ liệu thành từng batch
# minimal lize , each time only learn 64 model
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Đã tạo DataLoader thành công! Số lượng lô (batch) trong tập train: {len(train_loader)}")

In [ ]:
import torch
import torch.optim as optim

# 1. Chọn thiết bị tính toán: Ưu tiên GPU (Cuda) nếu có, không thì dùng CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Mô hình sẽ được chạy trên: {device}")

# 2. Khởi tạo mô hình và đẩy lên thiết bị đã chọn
input_dim = X_train_tfidf.shape[1]  # Số lượng đặc trưng từ TF-IDF (thường là 5000 như bạn thiết lập)
hidden_dim = 256                   # Số lượng nơ-ron ở tầng ẩn
num_classes = len(label_encoder_prof.classes_)  # Số lượng nghề nghiệp cần phân loại

model = ProfessionClassifier(input_dim, hidden_dim, num_classes).to(device)

# 3. Định nghĩa Hàm mất mát (Loss Function)
criterion = nn.CrossEntropyLoss()

# 4. Định nghĩa Thuật toán tối ưu (Optimizer)
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training Loop


In [ ]:
epochs = 5  # Số lần mô hình học đi học lại toàn bộ tập dữ liệu

print("Bắt đầu quá trình huấn luyện AI...")
for epoch in range(epochs):
    model.train()  # Bật chế độ huấn luyện (kích hoạt tầng Dropout để chống học vẹt)
    running_loss = 0.0

    # Duyệt qua từng lô (batch) dữ liệu từ băng chuyền train_loader
    for batch_x, batch_y, batch_g in train_loader:

        # Đẩy dữ liệu của batch đó lên GPU (phải cùng thiết bị với model)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # -----------------------------
        # BỐN BƯỚC THẦN THÁNH CỦA PYTORCH
        # -----------------------------

        # Bước 1: Forward Pass - Đưa văn bản vào bộ não AI để lấy dự đoán
        outputs = model(batch_x)

        # Bước 2: Tính toán độ sai lệch (Loss) giữa dự đoán và thực tế
        loss = criterion(outputs, batch_y)

        # Bước 3: Xóa sạch các vết tính toán (Gradient) của lượt cũ
        optimizer.zero_grad()

        # Bước 4: Backward Pass - Tính toán xem mỗi nơ-ron phải chịu trách nhiệm bao nhiêu cho lỗi sai này
        loss.backward()

        # Bước 5: Cập nhật trọng số - Người thầy Adam ra tay sửa đổi các nơ-ron
        optimizer.step()

        # Cộng dồn loss để theo dõi
        running_loss += loss.item()

    # Kết thúc 1 epoch, tính Loss trung bình
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] ---> Loss trung bình trên tập Train: {epoch_loss:.4f}")

print("Huấn luyện hoàn tất 100%!")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import numpy as np

# Bước 1: Chuyển mô hình sang chế độ đánh giá (Evaluation Mode)
model.eval()

# Khởi tạo các danh sách trống để gom kết quả từ tất cả các batch
all_preds = []
all_labels = []
all_genders = []

# Bước 2: Tắt tính năng tính đạo hàm (Gradient) để tiết kiệm RAM/GPU và chạy nhanh hơn
with torch.no_grad():
    for batch_x, batch_y, batch_g in test_loader:
        # Đẩy dữ liệu văn bản lên GPU để bộ não AI xử lý
        batch_x = batch_x.to(device)

        # Cho văn bản chạy qua mô hình để lấy kết quả dự đoán (dạng xác suất thô - logits)
        outputs = model(batch_x)

        # Tìm xem nghề nghiệp nào có điểm số xác suất cao nhất
        _, predicted = torch.max(outputs, 1)

        # Đưa các kết quả dự đoán từ GPU về lại CPU và gom vào danh sách tổng
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.numpy())
        all_genders.extend(batch_g.numpy())

print("--- ĐÃ THU THẬP XONG KẾT QUẢ TỪ TẬP TEST ---")

# Bước 3: Đổi các con số mã hóa (0, 1, 2...) về lại tên nghề nghiệp thực tế để dễ đọc báo cáo
true_professions = label_encoder_prof.inverse_transform(all_labels)
pred_professions = label_encoder_prof.inverse_transform(all_preds)
gender_labels = label_encoder_gender.inverse_transform(all_genders)

# In báo cáo phân loại truyền thống tổng thể
print("\n[1] BÁO CÁO HIỆU NĂNG TỔNG THỂ (CLASSIFICATION REPORT):")
print(classification_report(true_professions, pred_professions))

# Bước 4: Tạo DataFrame để tính toán các chỉ số Công bằng (Fairness Metrics)
results_df = pd.DataFrame({
    'True_Profession': true_professions,
    'Predicted_Profession': pred_professions,
    'Gender': gender_labels
})

# Tính Accuracy tổng thể
overall_acc = accuracy_score(results_df['True_Profession'], results_df['Predicted_Profession'])
print(f"Overall Accuracy: {overall_acc:.4f}")

# Bước 5: Tính toán Độ chính xác theo từng nhóm Giới tính (Subgroup Accuracy)
print("\n[2] ĐỘ CHÍNH XÁC CHIA THEO GIỚI TÍNH (SUBGROUP ACCURACY):")
subgroup_acc = results_df.groupby('Gender').apply(
    lambda x: accuracy_score(x['True_Profession'], x['Predicted_Profession'])
)
print(subgroup_acc)

# Bước 6: Tính Khoảng cách thiên vị (Accuracy Disparity)
acc_disparity = abs(subgroup_acc.iloc[0] - subgroup_acc.iloc[1])
print(f"\n[3] KHOẢNG CÁCH THIÊN VỊ (ACCURACY DISPARITY): {acc_disparity:.4f}")

In [ ]:
import re

def counterfactual_augmentation(text):
    # Tạo từ điển các từ cần tráo đổi
    replacement_dict = {
        r'\bhe\b': 'she', r'\bshe\b': 'he',
        r'\bhis\b': 'her', r'\bher\b': 'his',
        r'\bhim\b': 'her',
        r'\bman\b': 'woman', r'\bwoman\b': 'man',
        r'\bboy\b': 'girl', r'\bgirl\b': 'boy'
    }

    # Hàm phụ để xử lý tráo đổi không bị trùng lặp chéo
    def replace_match(match):
        word = match.group(0).lower()
        # Tìm từ thay thế tương ứng
        for key, value in replacement_dict.items():
            if re.match(key, word):
                return value
        return word

    # Tìm và thay thế tất cả các từ khóa xuất hiện trong text
    pattern = re.compile('|'.join(replacement_dict.keys()), re.IGNORECASE)
    return pattern.sub(replace_match, text)

# Chạy thử nghiệm để kiểm tra hàm hoạt động đúng không
sample_text = "he is a talented engineer and his project is great"
print("Trước:", sample_text)
print("Sau:  ", counterfactual_augmentation(sample_text))


In [ ]:
# Tạo một bản sao của tập train
augmented_df = train_df.copy()

# Áp dụng hàm tráo đổi giới tính lên cột văn bản
augmented_df['hard_text'] = augmented_df['hard_text'].apply(counterfactual_augmentation)

# Đảo ngược nhãn giới tính tương ứng (để dữ liệu logic)
augmented_df['gender'] = augmented_df['gender'].apply(lambda x: 'M' if x == 'F' else 'F')

# Gộp tập dữ liệu gốc và tập dữ liệu đã tráo đổi lại làm một
debiased_train_df = pd.concat([train_df, augmented_df], ignore_index=True)

print(f"Số lượng dữ liệu tập Train cũ: {len(train_df)}")
print(f"Số lượng dữ liệu tập Train mới sau khi tăng cường (CDA): {len(debiased_train_df)}")


In [ ]:
# Lưu tập dữ liệu đã khử bias thành file CSV
debiased_train_df.to_csv("biosbias_train_debiased.csv", index=False)